# Olist SQL Analysis

SQL analysis using DuckDB directly on raw CSV files.

## Data Schema

![Olist Schema](../images/ERD.png)

In [1]:
import duckdb
import pandas as pd

duckdb.sql("CREATE OR REPLACE VIEW orders AS SELECT * FROM '../data/raw/olist_orders_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW order_items AS SELECT * FROM '../data/raw/olist_order_items_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW payments AS SELECT * FROM '../data/raw/olist_order_payments_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW products AS SELECT * FROM '../data/raw/olist_products_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW customers AS SELECT * FROM '../data/raw/olist_customers_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW sellers AS SELECT * FROM '../data/raw/olist_sellers_dataset.csv'")
duckdb.sql("CREATE OR REPLACE VIEW translations AS SELECT * FROM '../data/raw/product_category_name_translation.csv'")
duckdb.sql("CREATE OR REPLACE VIEW reviews AS SELECT * FROM '../data/raw/olist_order_reviews_dataset.csv'")

In [2]:
for name in ['orders', 'order_items', 'payments', 'products', 'customers', 'sellers', 'translations', 'reviews']:
    print(name)
    display(duckdb.sql(f"SELECT * FROM {name} LIMIT 2").df())

orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


order_items


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


payments


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


translations


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories


reviews


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13


In [3]:
duckdb.sql("""
    CREATE OR REPLACE VIEW order_base AS
    SELECT
        o.order_id,
        c.customer_unique_id,
        c.customer_state,
        o.order_purchase_timestamp AS purchased_at,
        o.order_delivered_customer_date AS delivered_at,
        o.order_estimated_delivery_date + INTERVAL 1 DAY AS promised_at,
        date_diff('second', o.order_purchase_timestamp, o.order_delivered_customer_date) / 86400.0 AS delivery_days,
        (o.order_delivered_customer_date > o.order_estimated_delivery_date + INTERVAL 1 DAY)::INT AS is_late
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
""")

## 1. Data Quality

Checking for duplicates, missing values, and data integrity issues
before analysis.

### Q1: Duplicate order_id check

In [4]:
duckdb.sql("""
    WITH t AS(
    SELECT order_id, order_purchase_timestamp,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_purchase_timestamp) AS rnk
    FROM orders
    )

    SELECT order_id, order_purchase_timestamp
    FROM t
    WHERE rnk > 1
""").df()

,order_id,order_purchase_timestamp


### Q2: Order status distribution
96,478 out of 99,441 orders delivered (97%).

In [5]:
duckdb.sql("""
    SELECT order_status,
       COUNT(*) AS orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM orders
        GROUP BY order_status
    ORDER BY orders DESC
""").df()

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### Q3: Missing delivery dates in delivered orders
8 orders carry status delivered with no delivery date.
`order_base` drops them: 96,478 to 96,470.

In [6]:
duckdb.sql("""
    SELECT
        COUNT(*) AS delivered_status,
        COUNT(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS no_delivery_date,
        COUNT(*) FILTER (WHERE order_delivered_customer_date IS NOT NULL) AS usable
    FROM orders
    WHERE order_status = 'delivered'
""").df()

,delivered_status,no_delivery_date,usable
0,96478,8,96470


### Q4: Impossible dates (delivery before purchase)
0 orders found.

In [7]:
duckdb.sql("""
    SELECT COUNT(*) AS total_wrong_dates
    FROM orders
    WHERE order_delivered_customer_date < order_purchase_timestamp
""").df()

,total_wrong_dates
0,0


### Q5: Order vs order-item grain
Joining `order_items` multiplies rows. An order with three items becomes three.
Late count is 6,534 by order and 7,264 by item row.
Every query below reads from `order_base`. Seller aggregates go through DISTINCT order_id.

In [8]:
duckdb.sql("""
    SELECT
        COUNT(DISTINCT ob.order_id) AS orders,
        COUNT(*) AS order_item_rows,
        COUNT(DISTINCT ob.order_id) FILTER (WHERE ob.is_late = 1) AS late_orders,
        COUNT(*) FILTER (WHERE ob.is_late = 1) AS late_order_item_rows
    FROM order_base ob
    JOIN order_items oi ON ob.order_id = oi.order_id
""").df()

,orders,order_item_rows,late_orders,late_order_item_rows
0,96470,110189,6534,7264


## 2. Commercial Analysis (payments + items + products)

### Q6: Monthly revenue and MoM growth
December 2016 has R$19.62 in revenue, which makes January show +649,979%.
That is the platform launch, not growth. The series starts at 2017-01.

In [9]:
duckdb.sql("""
    WITH monthly AS (
        SELECT DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
               SUM(p.payment_value) AS revenue
        FROM orders o
        JOIN payments p
            ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
            AND o.order_purchase_timestamp >= DATE '2017-01-01'
        GROUP BY 1
    ),
    with_lag AS (
        SELECT month, revenue,
               LAG(revenue) OVER (ORDER BY month) AS prev_revenue
        FROM monthly
    )

    SELECT month,
           ROUND(revenue, 2) AS revenue,
           ROUND(prev_revenue, 2) AS prev_revenue,
           ROUND((revenue - prev_revenue) * 100.0 / prev_revenue, 2) AS mom_pct
    FROM with_lag
    ORDER BY month;
""").df()

,month,revenue,prev_revenue,mom_pct
0,2017-01-01,127545.67,NaN,NaN
1,2017-02-01,271298.65,127545.67,112.71
2,2017-03-01,414369.39,271298.65,52.74
3,2017-04-01,390952.18,414369.39,-5.65
4,2017-05-01,567066.73,390952.18,45.05
5,2017-06-01,490225.60,567066.73,-13.55
6,2017-07-01,566403.93,490225.60,15.54
7,2017-08-01,646000.61,566403.93,14.05
8,2017-09-01,701169.99,646000.61,8.54
9,2017-10-01,751140.27,701169.99,7.13


### Q7: Top 10 categories by revenue
Health & beauty leads with R$1.23M, followed by watches and bed/bath.

In [10]:
duckdb.sql("""
    SELECT t.product_category_name_english AS category,
       ROUND(SUM(oi.price), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN translations t ON p.product_category_name = t.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY 1
    ORDER BY revenue DESC
    LIMIT 10
""").df()

,category,revenue
0,health_beauty,1233131.72
1,watches_gifts,1166176.98
2,bed_bath_table,1023434.76
3,sports_leisure,954852.55
4,computers_accessories,888724.61
5,furniture_decor,711927.69
6,housewares,615628.69
7,cool_stuff,610204.10
8,auto,578966.65
9,toys,471286.48


### Q8: Payment method distribution
Credit card dominates at 73.9%. Boleto (Brazilian bank slip) is second at 19%.

In [11]:
duckdb.sql("""
    SELECT payment_type, COUNT(*) AS total, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS perc
    FROM payments
    GROUP BY payment_type
    ORDER BY total DESC
""").df()

,payment_type,total,perc
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00


### Q9: Average order value by state
Remote states show highest AOV despite low order volume. Buyers order high-value items to justify shipping costs.

In [12]:
duckdb.sql("""
    WITH order_totals AS (
        SELECT order_id, SUM(payment_value) AS order_value
        FROM payments
        GROUP BY order_id
    )

    SELECT c.customer_state, COUNT(*) AS orders,
           ROUND(AVG(ot.order_value), 2) AS avg_order_value
    FROM orders o
    JOIN order_totals ot ON o.order_id = ot.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    ORDER BY avg_order_value DESC;
""").df()

,customer_state,orders,avg_order_value
0,PB,517,266.60
1,AC,80,244.83
2,AP,67,240.92
3,AL,397,237.27
4,RO,243,234.47
5,PA,946,224.13
6,PI,476,221.16
7,RR,41,220.48
8,TO,274,219.01
9,RN,474,212.51


### Q10: Pareto, seller revenue concentration
556 of 3,095 sellers generate 80% of revenue.

In [13]:
duckdb.sql("""
    WITH seller_revenue AS(
        SELECT seller_id, SUM(price) AS revenue
        FROM order_items
        GROUP BY seller_id
    ), com AS(
        SELECT seller_id, ROUND(SUM (revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER(), 2) AS perc
        FROM seller_revenue
    )

    SELECT COUNT(seller_id)
    FROM com
    WHERE perc <= 0.8
""").df()

,count(seller_id)
0,556


## 3. Delivery & Reviews

Key findings from Python notebooks reproduced in SQL.

### Q11: On-time delivery rate
93.23% of delivered orders arrive by the end of the promised day.
Matches notebook 01. The promised date is stored at 00:00, so it is read as end of day.

In [14]:
duckdb.sql("""
    SELECT
        COUNT(*) AS orders,
        ROUND(AVG(1 - is_late) * 100, 2) AS on_time_pct
    FROM order_base
""").df()

,orders,on_time_pct
0,96470,93.23


### Q12: Review score by delivery status
Late orders average 2.27 stars, on-time 4.29.
The 6,381 late orders here match notebook 02 exactly. The 646 orders with no review drop out on the join.
Some orders carry more than one review, so the latest one is kept.

In [15]:
duckdb.sql("""
    WITH last_review AS (
        SELECT order_id, review_score,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_creation_date DESC) AS rnk
        FROM reviews
    )

    SELECT
        CASE WHEN ob.is_late = 1 THEN 'late' ELSE 'on-time' END AS delivery_status,
        COUNT(*) AS orders,
        ROUND(AVG(r.review_score), 2) AS avg_score
    FROM order_base ob
    JOIN last_review r ON ob.order_id = r.order_id AND r.rnk = 1
    GROUP BY 1
    ORDER BY avg_score
""").df()

,delivery_status,orders,avg_score
0,late,6381,2.27
1,on-time,89443,4.29


### Q13: Late delivery concentration by seller
50 of 2,970 sellers account for 2,495 of 6,547 late orders, 38.11%.
That is 1.7% of sellers behind 38% of late deliveries.
An order split across several sellers counts once for each, so the total runs 13 above the 6,534 from Q5.

In [16]:
duckdb.sql("""
    WITH order_seller AS (
        SELECT DISTINCT ob.order_id, oi.seller_id, ob.is_late
        FROM order_base ob
        JOIN order_items oi ON ob.order_id = oi.order_id
    ),
    seller_late AS (
        SELECT seller_id, COUNT(*) AS orders, SUM(is_late) AS late_orders
        FROM order_seller
        GROUP BY seller_id
    ),
    ranked AS (
        SELECT seller_id, late_orders,
               ROW_NUMBER() OVER (ORDER BY late_orders DESC) AS rnk
        FROM seller_late
    )
    SELECT
        COUNT(*) AS sellers,
        SUM(late_orders) FILTER (WHERE rnk <= 50) AS top50_late,
        SUM(late_orders) AS total_late,
        ROUND(SUM(late_orders) FILTER (WHERE rnk <= 50) * 100.0 / SUM(late_orders), 2) AS pct
    FROM ranked
""").df()

,sellers,top50_late,total_late,pct
0,2970,2495.0,6547.0,38.11


### Q14: Repeat buyers
2,997 of 96,096 customers placed more than one order, 3.12%.

In [17]:
duckdb.sql("""
    SELECT
        COUNT(*) AS customers,
        COUNT(*) FILTER (WHERE orders > 1) AS repeat_buyers,
        ROUND(COUNT(*) FILTER (WHERE orders > 1) * 100.0 / COUNT(*), 2) AS pct
    FROM (
        SELECT c.customer_unique_id, COUNT(*) AS orders
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_unique_id
    )
""").df()

,customers,repeat_buyers,pct
0,96096,2997,3.12


## 4. Cohorts and Retention

Repeat rate is 3.12%. This section tracks where those customers come from.

### Q15: Cohort activity by month
One row per cohort and month offset. Rows with month_index = 0 are cohort sizes.

In [18]:
cohorts = duckdb.sql("""
    WITH first_order AS (
        SELECT customer_unique_id, MIN(purchased_at) AS first_purchase
        FROM order_base
        GROUP BY customer_unique_id
    ), activity AS (
        SELECT ob.customer_unique_id,
               DATE_TRUNC('month', f.first_purchase) AS cohort_month,
               date_diff('month', DATE_TRUNC('month', f.first_purchase), DATE_TRUNC('month', ob.purchased_at)) AS month_index
        FROM order_base ob
        JOIN first_order f ON ob.customer_unique_id = f.customer_unique_id
    )
    SELECT cohort_month, month_index, COUNT(DISTINCT customer_unique_id) AS customers
    FROM activity
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

In [19]:
matrix = (cohorts
          .pivot(index='cohort_month', columns='month_index', values='customers')
          .fillna(0)
          .astype(int))
matrix.index = matrix.index.astype(str).str[:7]
matrix.iloc[:, :8]

month_index,0,1,2,3,4,5,6,7
cohort_month,,,,,,,,
2016-09,1,0,0,0,0,0,0,0
2016-10,262,0,0,0,0,0,1,0
2016-12,1,1,0,0,0,0,0,0
2017-01,717,2,2,1,3,1,3,1
2017-02,1628,3,5,2,7,2,4,3
2017-03,2503,11,9,10,9,4,4,8
2017-04,2256,14,5,4,6,6,8,7
2017-05,3450,16,16,10,10,11,14,5
2017-06,3037,15,12,13,9,12,11,7


In [20]:
retention = matrix.div(matrix[0], axis=0).mul(100).round(2)
retention.loc['2017-01':].iloc[:, :8]

month_index,0,1,2,3,4,5,6,7
cohort_month,,,,,,,,
2017-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14
2017-02,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18
2017-03,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32
2017-04,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31
2017-05,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14
2017-06,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23
2017-07,100.0,0.53,0.35,0.24,0.29,0.21,0.32,0.11
2017-08,100.0,0.69,0.35,0.27,0.35,0.52,0.30,0.27
2017-09,100.0,0.70,0.55,0.27,0.45,0.22,0.22,0.25
